# Financial Market Data Collection & Preprocessing

## Objective

The objective of this notebook is to build a clean and structured financial dataset for future quantitative finance research tasks including:

- Cointegration analysis
- Engle-Granger testing
- Johansen testing
- Spread analysis
- Rolling window stability analysis
- Statistical arbitrage research

The dataset consists of major US equities from the S&P 500 along with sector and market ETFs downloaded using Yahoo Finance.

## Import Libraries 

In [33]:
import yfinance as yf
import pandas as pd
import numpy as np

import requests
from bs4 import BeautifulSoup
from io import StringIO

In [34]:
# stocks = [

#     # Technology
#     'AAPL','MSFT','NVDA','GOOG','META','AMZN','TSLA',
#     'AMD','INTC','CSCO','ORCL','CRM','ADBE','QCOM',
#     'TXN','AVGO','IBM','AMAT','MU','NOW',

#     # Banking & Finance
#     'JPM','BAC','WFC','GS','MS','C','AXP','BLK',
#     'SCHW','USB','PNC','TFC',

#     # Energy
#     'XOM','CVX','COP','SLB','EOG','PSX','MPC',

#     # Healthcare
#     'JNJ','PFE','MRK','ABBV','TMO','LLY','ABT',
#     'DHR','BMY','UNH','CVS',

#     # Consumer
#     'KO','PEP','PG','COST','WMT','HD','MCD',
#     'SBUX','NKE','LOW','TGT',

#     # Industrials
#     'CAT','BA','HON','GE','MMM','UPS','FDX',
#     'LMT','DE',

#     # Telecom & Media
#     'VZ','T','CMCSA','DIS','NFLX',

#     # Utilities
#     'NEE','DUK','SO',

#     # Real Estate
#     'AMT','PLD','SPG',

#     # Materials
#     'LIN','APD','FCX',

#     # Transportation
#     'UNP','CSX',

#     # Semiconductor
#     'NXPI','KLAC','LRCX',

#     # Additional large caps
#     'PYPL','UBER','SHOP','ZM'
# ]

## Retrieve top 100 S&P 500 Companies 
The list of companies was extracted from the S&P 500 Wikipedia dataset instead of manually specifying individual stock names. This approach ensures a more systematic, scalable, and reproducible selection of top US market companies.


In [35]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')

table = soup.find('table', {'id': 'constituents'})

sp500 = pd.read_html(
    StringIO(str(table))
)[0]

sp500.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


## Selection and Formatting of Stock Symbols

The stock symbols of the first 100 S&P 500 companies are extracted from the dataset to create the research universe for analysis. Certain ticker symbols containing periods (.) are converted to hyphen (-) format to ensure compatibility with Yahoo Finance data retrieval conventions.


In [36]:
stocks = sp500['Symbol'].tolist()[:100]

stocks = [stock.replace('.', '-') for stock in stocks]

stocks[:100]

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE',
 'AEP',
 'AXP',
 'AIG',
 'AMT',
 'AWK',
 'AMP',
 'AME',
 'AMGN',
 'APH',
 'ADI',
 'AON',
 'APA',
 'APO',
 'AAPL',
 'AMAT',
 'APP',
 'APTV',
 'ACGL',
 'ADM',
 'ARES',
 'ANET',
 'AJG',
 'AIZ',
 'T',
 'ATO',
 'ADSK',
 'ADP',
 'AZO',
 'AVB',
 'AVY',
 'AXON',
 'BKR',
 'BALL',
 'BAC',
 'BAX',
 'BDX',
 'BRK-B',
 'BBY',
 'TECH',
 'BIIB',
 'BLK',
 'BX',
 'XYZ',
 'BNY',
 'BA',
 'BKNG',
 'BSX',
 'BMY',
 'AVGO',
 'BR',
 'BRO',
 'BF-B',
 'BLDR',
 'BG',
 'BXP',
 'CHRW',
 'CDNS',
 'CPT',
 'CPB',
 'COF',
 'CAH',
 'CCL',
 'CARR',
 'CVNA',
 'CASY',
 'CAT',
 'CBOE',
 'CBRE',
 'CDW',
 'COR',
 'CNC',
 'CNP',
 'CF',
 'CRL',
 'SCHW']

## Add Major ETFs

Major ETFs are manually included to provide representation of broad market indices and sector-specific benchmarks that are not part of the S&P 500 company list. These ETFs help capture overall market behavior, sector movements, and diversified market exposure for future correlation and cointegration analysis.


In [37]:
etfs = [
    'SPY','QQQ','DIA','IWM',
    'XLF','XLK','XLE','XLV',
    'XLY','XLP','XLI','XLU',
    'VNQ','GLD','TLT'
]

In [38]:
symbols = stocks + etfs

len(symbols)

115

## Historical Market Data Collection

Historical price data is downloaded using the Yahoo Finance API through the yfinance library. The dataset includes daily market data from 2018 onward, while the end date is dynamically generated.
The dataset begins in 2018 to ensure sufficient historical coverage across multiple market regimes, including:

- Bull market periods
- COVID-19 market crash
- Recovery phases
- Inflationary environments


In [39]:
start_date = '2018-01-01'

# Dynamic latest date
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')

data = yf.download(
    symbols,
    start=start_date,
    end=end_date,
    auto_adjust=False
)

[*********************100%***********************]  115 of 115 completed


## Extraction of Adjusted Closing Prices

Adjusted closing prices are extracted from the dataset as they account for stock splits and dividends, making them more reliable for accurate financial and time series analysis.


In [40]:
adj_close = data['Adj Close']

adj_close.head()

Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,VNQ,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XYZ
2018-01-02,63.522770,40.267078,68.773140,NaN,50.506077,27.988113,135.558243,177.699997,77.602272,31.201906,...,60.171005,25.747259,23.884136,66.254829,29.872921,45.314571,20.132578,72.723343,46.275242,36.169998
2018-01-03,65.139030,40.260063,69.849373,NaN,50.617760,28.013470,136.183868,181.039993,78.564995,30.960636,...,59.996315,26.132858,24.012451,66.611702,30.122097,45.298538,19.974419,73.419182,46.487701,37.310001
2018-01-04,64.650391,40.447075,69.451019,NaN,50.531849,28.118069,137.796463,183.220001,78.479065,31.482088,...,58.962769,26.290602,24.234875,67.099136,30.274368,45.426762,19.808550,73.523560,46.640114,38.099998
2018-01-05,65.684052,40.907570,70.660042,NaN,50.677902,28.007132,138.933090,185.339996,78.797112,31.271950,...,58.991886,26.280085,24.303308,67.560455,30.592758,45.627132,19.800837,74.149818,47.009621,41.139999
2018-01-08,65.825005,40.755638,69.527901,NaN,50.531849,28.010302,140.043411,185.039993,78.934639,31.201906,...,59.297588,26.437824,24.269093,67.838959,30.708111,45.739342,19.985998,73.880180,47.065041,40.759998


## Verification of Downloaded Symbols

The downloaded symbols are verified against the expected symbol list to identify missing or failed ticker downloads and ensure dataset completeness.


In [41]:
print("Expected Symbols:", len(symbols))

print("Downloaded Symbols:", len(adj_close.columns))

missing_symbols = set(symbols) - set(adj_close.columns)

if missing_symbols:
    print("Warning: Missing symbols detected")
    print(missing_symbols)
else:
    print("All symbols downloaded successfully")

Expected Symbols: 115
Downloaded Symbols: 115
All symbols downloaded successfully


## Save Raw Market Dataset

The complete raw market dataset is saved to preserve all downloaded fields for future research tasks including volatility analysis, liquidity analysis, and spread studies.

In [11]:
data.to_csv(
    '../data/raw/full_market_data.csv'
)

## Verify Datetime Index

The index is converted to datetime format to support rolling-window analysis, plotting, and statistical testing.

In [46]:
adj_close.index = pd.to_datetime(adj_close.index)

adj_close.index

DatetimeIndex(['2018-01-02', '2018-01-03', '2018-01-04', '2018-01-05',
               '2018-01-08', '2018-01-09', '2018-01-10', '2018-01-11',
               '2018-01-12', '2018-01-16',
               ...
               '2026-05-11', '2026-05-12', '2026-05-13', '2026-05-14',
               '2026-05-15', '2026-05-18', '2026-05-19', '2026-05-20',
               '2026-05-21', '2026-05-22'],
              dtype='datetime64[s]', length=2109, freq=None)

## Check for Duplicate Dates

Duplicate dates are identified and removed to maintain dataset consistency and ensure accurate time series analysis.


In [12]:
print(
    "Duplicate Dates:",
    adj_close.index.duplicated().sum()
)

adj_close = adj_close[
    ~adj_close.index.duplicated()
]

Duplicate Dates: 0


## Check for any missing value


In [48]:
adj_close.isnull().sum()

Ticker
A         0
AAPL      0
ABBV      0
ABNB    741
ABT       0
       ... 
XLP       0
XLU       0
XLV       0
XLY       0
XYZ       0
Length: 115, dtype: int64

## Missing value Percentage


In [13]:
missing_pct = (
    adj_close.isnull().mean() * 100
)

missing_pct.sort_values(
    ascending=False
)

Ticker
APP     39.165481
ABNB    35.135135
CARR    26.363205
A        0.000000
CNP      0.000000
          ...    
APA      0.000000
AOS      0.000000
AON      0.000000
ANET     0.000000
XYZ      0.000000
Length: 115, dtype: float64

## Filtering Assets with Excessive Missing Values

Assets containing more than 5% missing observations are removed to maintain dataset consistency and improve the reliability of future financial analysis.
A 5% threshold is used to balance data completeness.

In [14]:
valid_cols = missing_pct[
    missing_pct < 5
].index

adj_close = adj_close[valid_cols]

## Handling Remaining Missing Values

Remaining missing values are filled using forward-fill and backward-fill methods to create a complete dataset for time series analysis.


In [51]:
adj_close = adj_close.ffill().bfill()

In [ ]:
# Final check for any remaining missing values
adj_close.isnull().sum().sum()

0

## Validation check 
check fir any zero or negative price values

In [17]:
(adj_close <= 0).sum().sum()

0

In [ ]:
# download the cleaned adjusted close data
adj_close.to_csv(
    '../data/cleaned/adj_close_cleaned.csv'
)

In [24]:
# Calculate daily returns
returns = adj_close.pct_change().dropna()

returns.head()

Ticker,A,AAPL,ABBV,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,...,VNQ,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XYZ
2018-01-03,0.025444,-0.000174,0.015649,0.002211,0.000906,0.004615,0.018796,0.012406,-0.007733,0.010863,...,-0.002903,0.014977,0.005373,0.005386,0.008341,-0.000354,-0.007856,0.009569,0.004591,0.031518
2018-01-04,-0.007501,0.004645,-0.005703,-0.001697,0.003734,0.011841,0.012042,-0.001094,0.016843,0.009552,...,-0.017227,0.006036,0.009262,0.007317,0.005055,0.002831,-0.008304,0.001421,0.003279,0.021174
2018-01-05,0.015988,0.011385,0.017408,0.002890,-0.003945,0.008249,0.011571,0.004053,-0.006675,-0.000592,...,0.000494,-0.000400,0.002824,0.006875,0.010517,0.004411,-0.000390,0.008518,0.007922,0.079790
2018-01-08,0.002146,-0.003714,-0.016022,-0.002882,0.000113,0.007991,-0.001619,0.001746,-0.002240,-0.003043,...,0.005182,0.006002,-0.001408,0.004123,0.003771,0.002459,0.009351,-0.003636,0.001179,-0.009237
2018-01-09,0.024553,-0.000115,0.007538,0.001700,-0.012900,0.003335,0.008971,-0.002069,0.003242,0.006953,...,-0.012888,-0.002519,0.007755,0.006415,-0.002554,-0.001402,-0.009844,0.011773,0.001963,0.002944


## Generation of Log Returns

Log returns are calculated to support quantitative financial analysis and time series modeling due to their favorable mathematical and statistical properties.


In [56]:
log_returns = np.log(
    adj_close / adj_close.shift(1)
).dropna()

## Price Normalization

Asset prices are normalized to a common starting value for meaningful comparison of price movements across different stocks during exploratory data analysis.


In [18]:
normalized_prices = (
    adj_close / adj_close.iloc[0]
)

normalized_prices.head()

Ticker,A,AAPL,ABBV,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,...,VNQ,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XYZ
2018-01-02,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2018-01-03,1.025444,0.999826,1.015649,1.002211,1.000906,1.004615,1.018796,1.012406,0.992267,1.010863,...,0.997097,1.014977,1.005373,1.005386,1.008341,0.999646,0.992144,1.009569,1.004591,1.031518
2018-01-04,1.017752,1.004470,1.009856,1.000510,1.004643,1.016511,1.031064,1.011298,1.008980,1.020519,...,0.979920,1.021103,1.014685,1.012743,1.013438,1.002476,0.983905,1.011004,1.007885,1.053359
2018-01-05,1.034024,1.015906,1.027436,1.003402,1.000680,1.024896,1.042994,1.015396,1.002245,1.019916,...,0.980404,1.020695,1.017550,1.019705,1.024096,1.006898,0.983522,1.019615,1.015870,1.137407
2018-01-08,1.036243,1.012133,1.010975,1.000510,1.000793,1.033086,1.041306,1.017169,1.000000,1.016812,...,0.985484,1.026821,1.016117,1.023909,1.027958,1.009374,0.992719,1.015907,1.017068,1.126901


In [20]:
normalized_prices.to_csv(
    '../data/processed/normalized_prices.csv'
)

## Rolling Volatility Estimation

Rolling 30-day volatility is calculated to analyze changes in asset risk and price variability over time for future financial and stability analysis. A 30-day window is used because it approximately represents one trading month, making it suitable for capturing short-term market risk dynamics.


In [25]:
rolling_volatility = (
    returns.rolling(30).std()
)

rolling_volatility.head()

Ticker,A,AAPL,ABBV,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,...,VNQ,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XYZ
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
rolling_volatility.to_csv(
    '../data/processed/rolling_volatility.csv'
)

## Save Sector Metadata
Sector information is extracted and saved to support sector-wise analysis, asset grouping, and future cointegration and relationship studies between economically related companies.

In [26]:
sector_info = sp500[
    ['Symbol', 'GICS Sector']
]

sector_info['Symbol'] = (
    sector_info['Symbol']
    .str.replace('.', '-', regex=False)
)

sector_info.to_csv(
    '../data/processed/sector_info.csv',
    index=False
)

sector_info.head()

,Symbol,GICS Sector
0,MMM,Industrials
1,AOS,Industrials
2,ABT,Health Care
3,ABBV,Health Care
4,ACN,Information Technology


In [61]:
returns.to_csv(
    '../data/processed/returns.csv'
)

log_returns.to_csv(
    '../data/processed/log_returns.csv'
)

In [62]:
returns.describe()

Ticker,A,AAPL,ABBV,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,...,VNQ,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XYZ
count,2108.000000,2108.000000,2108.000000,2108.000000,2108.000000,2108.000000,2108.000000,2108.000000,2108.000000,2108.000000,...,2108.000000,2108.000000,2108.000000,2108.000000,2108.000000,2108.000000,2108.000000,2108.000000,2108.000000,2108.000000
mean,0.000449,0.001151,0.000685,0.000381,0.000757,0.000291,0.000413,0.001008,0.000589,0.000528,...,0.000322,0.000596,0.000477,0.000542,0.000990,0.000345,0.000467,0.000403,0.000560,0.001008
std,0.018271,0.019220,0.016821,0.015558,0.018436,0.017825,0.022735,0.021604,0.017560,0.016014,...,0.013806,0.019827,0.014696,0.013410,0.016522,0.009804,0.012757,0.010976,0.014904,0.037539
min,-0.110117,-0.128647,-0.162524,-0.100389,-0.168752,-0.095881,-0.167932,-0.166149,-0.241971,-0.152526,...,-0.177277,-0.201412,-0.137093,-0.113442,-0.138140,-0.093956,-0.113577,-0.098610,-0.126686,-0.285615
25%,-0.008783,-0.007833,-0.007053,-0.007017,-0.007444,-0.007688,-0.009936,-0.010254,-0.007583,-0.006485,...,-0.006056,-0.008869,-0.005888,-0.005553,-0.007096,-0.004274,-0.005872,-0.004970,-0.006418,-0.018755
50%,0.000887,0.001176,0.001108,0.000686,0.001242,0.001090,0.001133,0.001183,0.000943,0.001170,...,0.000789,0.001036,0.000846,0.000970,0.001738,0.000557,0.001028,0.000571,0.001327,0.001367
75%,0.010168,0.010930,0.009150,0.008588,0.009826,0.009101,0.012255,0.012569,0.009636,0.008315,...,0.007232,0.010251,0.007512,0.007314,0.009787,0.005355,0.006972,0.006024,0.008446,0.020240
max,0.098394,0.153289,0.137673,0.109360,0.153577,0.128573,0.177193,0.183876,0.102695,0.118036,...,0.089967,0.160373,0.131566,0.126512,0.134257,0.085106,0.127934,0.077057,0.108880,0.261396


In [64]:
returns.max()

returns.min()

Ticker
A      -0.110117
AAPL   -0.128647
ABBV   -0.162524
ABT    -0.100389
ACGL   -0.168752
          ...   
XLP    -0.093956
XLU    -0.113577
XLV    -0.098610
XLY    -0.126686
XYZ    -0.285615
Length: 112, dtype: float64

## Dataset summary

In [65]:
print("FINAL DATASET SUMMARY")

print("-" * 40)

print(
    "Number of Assets:",
    adj_close.shape[1]
)

print(
    "Trading Days:",
    adj_close.shape[0]
)

print(
    "Start Date:",
    adj_close.index.min()
)

print(
    "End Date:",
    adj_close.index.max()
)

print(
    "Remaining Missing Values:",
    adj_close.isnull().sum().sum()
)

FINAL DATASET SUMMARY
----------------------------------------
Number of Assets: 112
Trading Days: 2109
Start Date: 2018-01-02 00:00:00
End Date: 2026-05-22 00:00:00
Remaining Missing Values: 0
